# Final test — Bangla XLM-R (three frozen seeds)

Inference only. The model and checkpoint hashes were selected using Stage 2 validation before test access. Keep `RUN_FINAL_TEST=False` while checking setup. Supply three Kaggle checkpoint input paths before enabling the run. No training or output overwriting is performed.


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/kaggle/working/Capstone-Project-final-test')
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'experimental-protocol',
        'https://github.com/iftekharuddin27/Capstone-Project.git', str(REPO_DIR)
    ], check=True)
required = REPO_DIR / 'corrected_pipeline' / 'final_test.py'
if not required.is_file():
    raise FileNotFoundError(
        f'{required} is missing. Push the new final-test files to the experimental-protocol branch, '
        'then start a fresh Kaggle session and run this cell again.'
    )
print('Repository ready:', REPO_DIR)
print('Inference-only evaluator found:', required)


In [ ]:
RUN_FINAL_TEST = False
LANGUAGE = 'bangla'
CHECKPOINT_PATHS = {
    42: '/kaggle/input/PASTE_BN_SEED42_CHECKPOINT_PATH/best_checkpoint_cpu.pt',
    123: '/kaggle/input/PASTE_BN_SEED123_CHECKPOINT_PATH/best_checkpoint_cpu.pt',
    2026: '/kaggle/input/PASTE_BN_SEED2026_CHECKPOINT_PATH/best_checkpoint_cpu.pt',
}

import subprocess, sys
sys.path.insert(0, str(REPO_DIR))
from corrected_pipeline.final_test import FROZEN_RUNS, frozen_run

assert set(CHECKPOINT_PATHS) == {42, 123, 2026}
for seed in (42, 123, 2026):
    frozen = frozen_run(LANGUAGE, seed, REPO_DIR)
    print(f'seed {seed} | checkpoint SHA-256: {frozen["checkpoint_sha256"]} | output: {frozen["output"]}')

if not RUN_FINAL_TEST:
    print('Preflight only: no checkpoint or test CSV has been opened.')
else:
    paths = {seed: Path(value) for seed, value in CHECKPOINT_PATHS.items()}
    for seed, path in paths.items():
        if 'PASTE_' in str(path) or not path.is_file():
            raise FileNotFoundError(f'Attach checkpoint seed {seed} and paste its real Kaggle input path: {path}')
    subprocess.run([sys.executable, '-m', 'compileall', '-q', 'corrected_pipeline', 'tests_stage1a'], cwd=REPO_DIR, check=True)
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests_stage1a', '-v'], cwd=REPO_DIR, check=True)
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a Kaggle GPU in Notebook Settings before final-test inference')
    for seed, checkpoint in paths.items():
        print(f'FINAL TEST — {LANGUAGE} seed {seed}; loading only the pinned checkpoint')
        subprocess.run([
            sys.executable, '-m', 'corrected_pipeline.final_test',
            '--language', LANGUAGE, '--seed', str(seed),
            '--checkpoint', str(checkpoint), '--repo', str(REPO_DIR),
        ], cwd=REPO_DIR, check=True)
    print('All three frozen {LANGUAGE} final-test runs completed.'.replace('{LANGUAGE}', LANGUAGE))


In [ ]:
if RUN_FINAL_TEST:
    import json, statistics, zipfile
    from pathlib import Path
    from corrected_pipeline.final_test import FINAL_STATUS
    root = Path('/kaggle/working')
    entries = {}
    for seed in (42, 123, 2026):
        folder = root / f'corrected_final_test_bn_xlmr_seed{seed}'
        metrics = json.loads((folder / 'metrics.json').read_text(encoding='utf-8'))
        provenance = json.loads((folder / 'provenance.json').read_text(encoding='utf-8'))
        if metrics['result_status'] != FINAL_STATUS or provenance['seed'] != seed or provenance['language'] != LANGUAGE:
            raise RuntimeError(f'Incomplete or mismatched final-test results: {folder}')
        entries[str(seed)] = {'macro_f1': metrics['macro_f1'], 'sarcasm_f1': metrics['sarcasm_f1'], 'hate_f1': metrics['hate_f1']}
    summary = {
        'result_status': FINAL_STATUS, 'language': LANGUAGE, 'model': 'xlm-roberta-base',
        'selection': 'preselected using Stage 2 validation macro-F1',
        'per_seed': entries,
        'mean_and_sample_sd': {
            metric: {'mean': statistics.mean(values), 'sample_sd': statistics.stdev(values)}
            for metric in ('macro_f1','sarcasm_f1','hate_f1')
            for values in [[entries[str(seed)][metric] for seed in (42, 123, 2026)]]
        },
    }
    summary_path = root / 'corrected_final_test_bn_xlmr_summary.json'
    summary_path.write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
    archive_path = root / 'corrected_final_test_bn_xlmr_results.zip'
    if archive_path.exists():
        raise FileExistsError(f'Refusing to overwrite a previous results ZIP: {archive_path}')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
        archive.write(summary_path, arcname=summary_path.name)
        for seed in (42, 123, 2026):
            folder = root / f'corrected_final_test_bn_xlmr_seed{seed}'
            for file in folder.iterdir():
                if file.is_file():
                    archive.write(file, arcname=str(file.relative_to(root)))
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise RuntimeError('ZIP integrity check failed')
    print(json.dumps(summary, indent=2))
    print('DOWNLOAD THIS RESULTS ZIP:', archive_path)
else:
    print('Results ZIP will be created after the three final-test runs complete.')
